# Communication Dropout Diagnostics

Read one or more JSON outputs from `scripts/diagnose_joint_happo.py` and compare the core joint performance metrics:

- scout recall: mean and std
- confirmation recall: mean and std
- final confidence: mean and std
- final coverage: mean and std

Add the diagnostic JSON files you want to compare to `JSON_FILES` in the first code cell.

In [ ]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()

# Add diagnose_joint_happo.py JSON outputs here.
JSON_FILES = [
    # PROJECT_ROOT / 'outputs/900k/joint_suvivor_900k_300_warmup_1000_1099.json',
]

# Optional convenience fallback while exploring interactively.
if not JSON_FILES:
    JSON_FILES = sorted((PROJECT_ROOT / 'outputs').glob('**/joint*.json'))

JSON_FILES = [Path(path) for path in JSON_FILES]
missing = [path for path in JSON_FILES if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing diagnostic JSON files:\n' + '\n'.join(str(p) for p in missing))
if not JSON_FILES:
    raise FileNotFoundError('No joint diagnostic JSON files configured or found under outputs/**/joint*.json')

print(f'Loaded file list: {len(JSON_FILES)} JSON file(s)')
for path in JSON_FILES:
    print(' -', path.relative_to(PROJECT_ROOT) if path.is_relative_to(PROJECT_ROOT) else path)

In [ ]:
def _summary_value(summary, *keys, default=float('nan')):
    for key in keys:
        if key in summary:
            return summary[key]
    return default


def _label_from_payload(path, payload):
    scenario = payload.get('scenario', {})
    dropout = scenario.get('comms_dropout', None)
    mode = scenario.get('comms_dropout_mode', None)
    if dropout is None:
        return path.stem
    return f'{path.stem} | dropout={dropout:g}, mode={mode or "?"}'


records = []
for path in JSON_FILES:
    payload = json.loads(path.read_text())
    summary = payload.get('summary', {})
    records.append({
        'file': str(path),
        'label': _label_from_payload(path, payload),
        'episodes': _summary_value(summary, 'episodes'),
        'scout_recall_mean': _summary_value(summary, 'mean_scout_recall'),
        'scout_recall_std': _summary_value(summary, 'std_scout_recall'),
        'confirm_recall_mean': _summary_value(summary, 'mean_confirm_recall'),
        'confirm_recall_std': _summary_value(summary, 'std_confirm_recall'),
        'final_confidence_mean': _summary_value(summary, 'mean_final_confidence'),
        'final_confidence_std': _summary_value(summary, 'std_final_confidence'),
        'final_coverage_mean': _summary_value(summary, 'mean_final_coverage_fraction'),
        'final_coverage_std': _summary_value(summary, 'std_final_coverage_fraction'),
    })

metrics = pd.DataFrame.from_records(records)
metrics

## Compact Table

The table below formats each metric as `mean +/- std` for quick comparison.

In [ ]:
def _pm(mean, std):
    if pd.isna(mean):
        return 'n/a'
    if pd.isna(std):
        return f'{mean:.3f}'
    return f'{mean:.3f} +/- {std:.3f}'


compact = pd.DataFrame({
    'label': metrics['label'],
    'episodes': metrics['episodes'].astype('Int64'),
    'scout recall': [
        _pm(mean, std)
        for mean, std in zip(metrics['scout_recall_mean'], metrics['scout_recall_std'])
    ],
    'confirm recall': [
        _pm(mean, std)
        for mean, std in zip(metrics['confirm_recall_mean'], metrics['confirm_recall_std'])
    ],
    'final confidence': [
        _pm(mean, std)
        for mean, std in zip(metrics['final_confidence_mean'], metrics['final_confidence_std'])
    ],
    'final coverage': [
        _pm(mean, std)
        for mean, std in zip(metrics['final_coverage_mean'], metrics['final_coverage_std'])
    ],
}).set_index('label')

compact